# Export the released Qwen3.5-4B joint RotQuant winner

This notebook reconstructs the matched-release winner, `uniform_w4__frozen_mixed_3.25`, exports its seed-0 weights as a self-contained packed checkpoint, embeds the validated frozen Gaussian K/V recipe in the manifest, and verifies the artifact in a fresh process. It does not repeat model selection or recovery trials.

## Goal

Produce the exact development-release artifact selected by the matched follow-up: uniform 4-bit Gaussian weights with FWHT rotation and the frozen mixed 3.25-bpv K/V deployment map. Completion requires a matching seed-0 perplexity, real file-byte accounting, no serialized fp fallback cache, and a successful fresh-process reload.

### Key assumptions and boundaries

- The matched release decision is an input, not something this notebook recomputes.
- The upstream model ID was not revision-pinned in the original matrix. This run records the current Hub revision and requires seed-0 WikiText-2 PPL to match the released row within a narrow tolerance.
- `patch.fallback=true` accelerates CUDA quality evaluation, but fallback tensors must not appear in the checkpoint.
- The K/V recipe is embedded as deployment metadata. `load_packed_model` restores packed weights; the serving runtime must separately read and enforce `manifest["deployment"]["kv_cache"]`.
- Artifact bytes are measured from files on disk. Runtime VRAM and throughput remain unverified because RotQuant still lacks a fused packed CUDA matmul.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-joint-winner-export")
MODEL_ID = "unsloth/Qwen3.5-4B"
CONFIG_RELATIVE_PATH = Path("configs/qwen35_4b_joint_cuda.yaml")

DRIVE_ROOT = Path("/content/drive/MyDrive/rotquant")
JOINT_RUN = "8d9676f109fa"
FOLLOWUP_RUN = "206ea94454b3"
FROZEN_KV_RUN = "49d3cf1182f0"
FOLLOWUP_SUMMARY_PATH = (
    DRIVE_ROOT / "qwen35_joint_release_followup" / FOLLOWUP_RUN
    / "joint_release_followup_summary.json"
)
FROZEN_KV_SUMMARY_PATH = (
    DRIVE_ROOT / "qwen35_kv_matrix/frozen_transfer" / FROZEN_KV_RUN
    / "kv_frozen_transfer_summary.json"
)
EXPORT_BASE = DRIVE_ROOT / "qwen35_joint_winner_export"
EXPORT_LABEL = "uniform_w4_frozen_mixed_3p25_seed0_v1"

EXPECTED_TRIAL = "uniform_w4__frozen_mixed_3.25"
EXPECTED_QUANTIZED_MODULES = 200
EXPECTED_EFFECTIVE_KV_BPV = 3.25
SEED0_PPL_ABS_TOLERANCE = 0.02
ARTIFACT_SIZE_RELATIVE_TOLERANCE = 0.01

CONFIRM_EXPORT = False  # Review every path and gate, then set True.
VERIFY_RELOAD = True
PUBLISH_TO_HUB = False
HF_REPO_ID = None  # Example: "your-name/qwen35-4b-rotquant-joint"
HF_REPO_PRIVATE = True

print({
    "repo_ref": REPO_REF,
    "winner": EXPECTED_TRIAL,
    "followup_summary": str(FOLLOWUP_SUMMARY_PATH),
    "confirm_export": CONFIRM_EXPORT,
})

### 1. Verify CUDA and mount Google Drive

In [ ]:
import os
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB")
if vram_gib < 40:
    print("WARNING: the quality-only fp16 fallback may OOM below 40 GiB.")
subprocess.run(["nvidia-smi"], check=True)

from google.colab import drive

drive.mount("/content/drive")
EXPORT_BASE.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

### 2. Fetch the exporter and install its runtime

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_DIR, check=True)

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
RESULT_ROOT = EXPORT_BASE / commit[:12]
CHECKPOINT_DIR = RESULT_ROOT / EXPORT_LABEL
RUN_OUTPUT_DIR = RESULT_ROOT / "export_run"
DEPLOYMENT_METADATA_PATH = RESULT_ROOT / "deployment_metadata.json"
EXPORT_REPORT_PATH = RESULT_ROOT / "winner_export_report.json"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print({"commit": commit, "checkpoint": str(CHECKPOINT_DIR)})

runtime_packages = [
    "transformers>=5.9,<6", "datasets>=4.8", "accelerate",
    "safetensors", "sentencepiece", "scipy", "pyyaml",
    "pandas", "huggingface_hub",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)
repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
import rotquant

print(f"rotquant import: {Path(rotquant.__file__).resolve()}")

### 3. Validate the release inputs and recipe provenance

This cell fails closed if the matched release status, frozen-map recommendation, seed-0 reference row, or weight-defining implementation has drifted.

In [ ]:
import json

import pandas as pd
import yaml
from huggingface_hub import HfApi

config_path = REPO_DIR / CONFIG_RELATIVE_PATH
required_paths = [config_path, FOLLOWUP_SUMMARY_PATH, FROZEN_KV_SUMMARY_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, "Missing release inputs: " + ", ".join(missing)

with config_path.open() as handle:
    base_config = yaml.safe_load(handle)
with FOLLOWUP_SUMMARY_PATH.open() as handle:
    followup_summary = json.load(handle)
with FROZEN_KV_SUMMARY_PATH.open() as handle:
    frozen_kv_summary = json.load(handle)

assert followup_summary["git_sha"].startswith(FOLLOWUP_RUN)
assert followup_summary["baseline_git_sha"].startswith(JOINT_RUN)
assert followup_summary["winner"] == EXPECTED_TRIAL
assert followup_summary["release_status"] == "matched_release_gates_passed"
assert followup_summary["unrecovered"]["release_gate_pass"] is True
assert followup_summary["recovered_promoted"] is False
assert followup_summary["recovery_confirmation_ran"] is False
assert frozen_kv_summary["recommendation"] == "mixed"
assert frozen_kv_summary["source_run_id"] == "bf94f2045a90"

frozen_recipe = frozen_kv_summary["mixed_recipe"]
assert [row["layer"] for row in frozen_recipe] == list(range(3, 32, 4))
assert all({"layer", "key_bits", "value_bits"} <= set(row) for row in frozen_recipe)
mean_payload_bits = sum(
    row["key_bits"] + row["value_bits"] for row in frozen_recipe
) / (2 * len(frozen_recipe))
assert mean_payload_bits == 3.0

release_validation_path = Path(
    followup_summary["baseline_artifacts"]["release_validation"]
)
assert release_validation_path.exists(), release_validation_path
release_validation = pd.read_csv(release_validation_path)
seed0_rows = release_validation[
    (release_validation["trial"] == EXPECTED_TRIAL)
    & (release_validation["seed"] == 0)
]
assert len(seed0_rows) == 1
seed0_reference = seed0_rows.iloc[0]
expected_seed0_ppl = float(seed0_reference["ppl"])
expected_weight_gb = float(seed0_reference["estimated_weight_GB"])
expected_weight_reduction = float(seed0_reference["weight_reduction"])

assert base_config["model"] == MODEL_ID
assert base_config["quant"] == {
    "bits": 4, "codebook": "gaussian", "scale": "mse_search",
    "group_size": 128, "error_comp": "none",
}
assert base_config["patch"]["rotation"] == "fwht"
assert base_config["patch"]["block"] == 128
assert base_config["patch"]["fallback"] is True

weight_defining_paths = [
    "rotquant/codebooks.py", "rotquant/quantize.py", "rotquant/pack.py",
    "rotquant/linear.py", "rotquant/patch.py", "rotquant/rotate.py",
    str(CONFIG_RELATIVE_PATH),
]
baseline_git_sha = followup_summary["baseline_git_sha"]
subprocess.run(["git", "cat-file", "-e", baseline_git_sha], cwd=REPO_DIR, check=True)
core_drift = subprocess.run(
    ["git", "diff", "--quiet", baseline_git_sha, commit, "--", *weight_defining_paths],
    cwd=REPO_DIR, check=False,
)
assert core_drift.returncode == 0, (
    "Weight-defining code changed since the selected matrix. Revalidate before export."
)
model_revision = HfApi().model_info(MODEL_ID).sha
print({
    "release_status": followup_summary["release_status"],
    "seed0_ppl": expected_seed0_ppl,
    "expected_weight_GB": expected_weight_gb,
    "frozen_layers": len(frozen_recipe),
    "model_revision": model_revision,
    "weight_code_drift": False,
})

### 4. Write the deployment contract

The exporter embeds this JSON object directly in `rotquant_config.json`. The exact K/V map therefore travels with the checkpoint rather than living only in a separate experiment directory.

In [ ]:
deployment_metadata = {
    "schema_version": 1,
    "recipe_id": EXPECTED_TRIAL,
    "release_status": followup_summary["release_status"],
    "weight": {
        "seed": 0,
        "bits": 4,
        "codebook": "gaussian",
        "scale": "mse_search",
        "group_size": 128,
        "rotation": "fwht",
        "rotation_block": 128,
        "dynamic": None,
        "expected_seed0_wikitext2_ppl": expected_seed0_ppl,
        "expected_complete_weight_GB": expected_weight_gb,
        "expected_weight_reduction": expected_weight_reduction,
    },
    "kv_cache": {
        "codebook": "gaussian",
        "group_size": 64,
        "effective_bpv": EXPECTED_EFFECTIVE_KV_BPV,
        "frozen_recipe": frozen_recipe,
        "runtime_contract": (
            "Validated recipe metadata only; the serving runtime must "
            "apply this map when allocating and quantizing K/V cache tensors."
        ),
    },
    "validation": {
        "source_ppl": followup_summary["source_ppl"],
        "mean_ppl": followup_summary["unrecovered"]["mean_ppl"],
        "worst_relative_ppl": followup_summary["unrecovered"]["worst_relative_ppl"],
        "mean_cache_kl_ratio_to_matched_k4v4": followup_summary["unrecovered"]["mean_cache_kl_ratio"],
        "worst_cache_kl_ratio_to_matched_k4v4": followup_summary["unrecovered"]["worst_cache_kl_ratio"],
    },
    "provenance": {
        "model_id": MODEL_ID,
        "model_revision_at_export": model_revision,
        "joint_matrix_git_sha": baseline_git_sha,
        "release_followup_git_sha": followup_summary["git_sha"],
        "exporter_git_sha": commit,
        "release_summary": str(FOLLOWUP_SUMMARY_PATH),
        "frozen_kv_summary": str(FROZEN_KV_SUMMARY_PATH),
    },
}
DEPLOYMENT_METADATA_PATH.write_text(
    json.dumps(deployment_metadata, indent=2) + "\n"
)
print(json.dumps(deployment_metadata, indent=2))

## Steps

### 5. Reconstruct, confirm PPL, and export

This is one deterministic seed-0 run. K/V evaluation, trajectory evaluation, and recovery training are disabled; the already-selected frozen map is carried as metadata. WikiText-2 PPL remains enabled as a reconstruction guard.

In [ ]:
import shlex

assert CONFIRM_EXPORT, (
    "Review the release inputs and output path, then set CONFIRM_EXPORT=True."
)
manifest_path = CHECKPOINT_DIR / "rotquant_config.json"
if manifest_path.exists():
    print(f"Reusing completed checkpoint: {CHECKPOINT_DIR}")
else:
    partial_files = list(CHECKPOINT_DIR.glob("*")) if CHECKPOINT_DIR.exists() else []
    assert not partial_files, (
        f"Checkpoint directory is partial or stale: {CHECKPOINT_DIR}. "
        "Inspect it and choose a new EXPORT_LABEL; this notebook will not delete it."
    )
    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    overrides = [
        "patch.enabled=true",
        "quant.bits=4",
        "patch.dynamic=null",
        "patch.rotation=fwht",
        "patch.train_rotation=null",
        "eval.perplexity=true",
        "eval.ppl.seq_len=256",
        "eval.ppl.max_samples=64",
        'eval.ppl_datasets=["wikitext2"]',
        "eval.kv_cache=false",
        "eval.trajectory=false",
        "eval.zeroshot=false",
        "eval.throughput=false",
        f"model_revision={model_revision}",
    ]
    command = [
        sys.executable, str(REPO_DIR / "scripts/run_experiment.py"),
        str(config_path),
        "--output-dir", str(RUN_OUTPUT_DIR),
        "--device", "cuda",
        "--seed", "0",
        "--export-dir", str(CHECKPOINT_DIR),
        "--export-processor",
        "--export-deployment-metadata", str(DEPLOYMENT_METADATA_PATH),
    ]
    for override in overrides:
        command.extend(["--set", override])
    print("Running:", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)
assert manifest_path.exists(), "Export did not complete; inspect the preceding log."

## Checks

### 6. Confirm the reconstruction and real artifact bytes

In [ ]:
result_files = sorted(
    RUN_OUTPUT_DIR.glob("*.json"), key=lambda path: path.stat().st_mtime
)
assert result_files, f"No export-run result JSON in {RUN_OUTPUT_DIR}"
export_result_path = result_files[-1]
with export_result_path.open() as handle:
    export_payload = json.load(handle)
metrics = export_payload["metrics"]
actual_ppl = float(metrics["ppl_wikitext2"])
ppl_abs_error = abs(actual_ppl - expected_seed0_ppl)
assert ppl_abs_error <= SEED0_PPL_ABS_TOLERANCE, (
    f"Seed-0 PPL drifted: expected {expected_seed0_ppl:.6f}, "
    f"got {actual_ppl:.6f}. Do not release this reconstruction."
)
assert abs(float(metrics["bits_per_weight_mean"]) - 4.125) < 1e-9
packed_checkpoint_metrics = metrics["packed_checkpoint"]
assert packed_checkpoint_metrics["fallback_cache_serialized"] is False
assert packed_checkpoint_metrics["quantized_modules"] == EXPECTED_QUANTIZED_MODULES
print({
    "result": str(export_result_path),
    "expected_seed0_ppl": expected_seed0_ppl,
    "actual_seed0_ppl": actual_ppl,
    "absolute_error": ppl_abs_error,
    "effective_weight_bpw": metrics["bits_per_weight_mean"],
})

### 7. Audit the manifest and reject serialized fallback state

In [ ]:
from safetensors import safe_open

with manifest_path.open() as handle:
    manifest = json.load(handle)
assert manifest["format"] == "rotquant-packed"
assert manifest["format_version"] == 1
assert manifest["base_model"] == MODEL_ID
assert manifest["base_model_revision"] == model_revision
assert manifest["model_loader"] == "multimodal_lm"
assert len(manifest["quantized_modules"]) == EXPECTED_QUANTIZED_MODULES
assert {row["rotation"]["kind"] for row in manifest["quantized_modules"]} == {"fwht"}
assert {row["qweight"]["packed"]["bits"] for row in manifest["quantized_modules"]} == {4}
assert {row["lora_rank"] for row in manifest["quantized_modules"]} == {0}
assert manifest["deployment"] == deployment_metadata
assert manifest["deployment"]["kv_cache"]["frozen_recipe"] == frozen_recipe

model_state_path = CHECKPOINT_DIR / manifest["model_state"]
packed_state_path = CHECKPOINT_DIR / manifest["packed_state"]
assert model_state_path.is_file() and packed_state_path.is_file()
forbidden_markers = ("_fp_cache", "fallback", "dequantized_weight", "source_weight")
tensor_keys = []
for state_path in (model_state_path, packed_state_path):
    with safe_open(state_path, framework="pt", device="cpu") as handle:
        tensor_keys.extend(handle.keys())
lower_keys = [key.lower() for key in tensor_keys]
bad_keys = [
    key for key in lower_keys if any(marker in key for marker in forbidden_markers)
]
assert not bad_keys, f"Forbidden fallback/source tensors found: {bad_keys[:10]}"
assert not any(marker in manifest_path.read_text().lower() for marker in forbidden_markers)

tensor_file_bytes = model_state_path.stat().st_size + packed_state_path.stat().st_size
expected_weight_bytes = expected_weight_gb * 1e9
size_relative_error = abs(tensor_file_bytes - expected_weight_bytes) / expected_weight_bytes
assert size_relative_error <= ARTIFACT_SIZE_RELATIVE_TOLERANCE, (
    f"Packed tensor bytes differ from the logical release estimate by "
    f"{size_relative_error:.2%}; investigate before release."
)
artifact_files = sorted(path for path in CHECKPOINT_DIR.rglob("*") if path.is_file())
artifact_bytes = sum(path.stat().st_size for path in artifact_files)
print({
    "tensor_file_GB": tensor_file_bytes / 1e9,
    "complete_artifact_GB": artifact_bytes / 1e9,
    "logical_estimate_GB": expected_weight_gb,
    "size_relative_error": size_relative_error,
    "tensor_keys_audited": len(tensor_keys),
    "fallback_cache_serialized": False,
})

### 8. Write checksums and an export report

In [ ]:
import hashlib
from datetime import datetime, timezone


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()

checksum_path = CHECKPOINT_DIR / "SHA256SUMS"
files_to_hash = [
    path for path in sorted(CHECKPOINT_DIR.rglob("*"))
    if path.is_file() and path != checksum_path
]
checksums = {
    str(path.relative_to(CHECKPOINT_DIR)): sha256_file(path)
    for path in files_to_hash
}
checksum_path.write_text(
    "".join(f"{digest}  {name}\n" for name, digest in checksums.items())
)
export_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "artifact_checks_passed",
    "checkpoint": str(CHECKPOINT_DIR),
    "exporter_git_sha": commit,
    "model_revision": model_revision,
    "recipe_id": EXPECTED_TRIAL,
    "seed0_ppl": actual_ppl,
    "seed0_ppl_abs_error": ppl_abs_error,
    "tensor_file_bytes": tensor_file_bytes,
    "artifact_bytes_before_checksums": artifact_bytes,
    "size_relative_error_to_release_estimate": size_relative_error,
    "quantized_modules": len(manifest["quantized_modules"]),
    "fallback_cache_serialized": False,
    "effective_kv_bpv": manifest["deployment"]["kv_cache"]["effective_bpv"],
    "checksums": checksums,
}
EXPORT_REPORT_PATH.write_text(json.dumps(export_report, indent=2) + "\n")
print(json.dumps({
    "report": str(EXPORT_REPORT_PATH),
    "checksums": str(checksum_path),
    "hashed_files": len(checksums),
}, indent=2))

### 9. Reload in a fresh process and generate one token

The loader runs with compressed persistent storage (`fallback=False`). The audit flag fails unless all 200 restored `QuantLinear` modules begin without fp fallback caches.

In [ ]:
if VERIFY_RELOAD:
    verify_command = [
        sys.executable, str(REPO_DIR / "scripts/generate_packed.py"),
        str(CHECKPOINT_DIR),
        "--device", "cuda",
        "--max-new-tokens", "1",
        "--prompt", "Rotation-aware quantization is",
        "--audit-no-fallback-cache",
    ]
    print("Running:", shlex.join(verify_command), flush=True)
    subprocess.run(verify_command, cwd=REPO_DIR, check=True)
    export_report["fresh_process_reload"] = "passed"
    EXPORT_REPORT_PATH.write_text(json.dumps(export_report, indent=2) + "\n")
    print("Packed checkpoint reload, fallback audit, and generation: PASS")
else:
    print("Reload verification skipped; artifact verification is incomplete.")

### 10. Optionally publish the complete directory

Publishing is disabled by default. Upload only after every preceding check passes, and keep the manifest, both safetensors files, processor/tokenizer assets, and checksum file together.

In [ ]:
if PUBLISH_TO_HUB:
    assert HF_REPO_ID, "Set HF_REPO_ID before publishing."
    from huggingface_hub import HfApi, notebook_login

    notebook_login()
    api = HfApi()
    api.create_repo(HF_REPO_ID, private=HF_REPO_PRIVATE, exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID, folder_path=CHECKPOINT_DIR, repo_type="model"
    )
    print(f"Uploaded https://huggingface.co/{HF_REPO_ID}")
else:
    print("Hub publication disabled; the verified checkpoint remains in Drive.")

## Next Steps

- Treat `winner_export_report.json` and `SHA256SUMS` as the release receipt.
- Make the serving adapter read `rotquant_config.json` and apply the embedded frozen K/V map; the current packed-weight loader does not activate it automatically.
- Before production, run broader task quality, long-context perplexity/retrieval, and native packed runtime memory/throughput validation.
- Do not quote the fallback process's VRAM as the packed deployment footprint.